# Token Cost Optimization with Claude API

**Practical techniques to reduce your Claude API token costs — with real before/after measurements.**

Every Claude API call costs money based on the number of input and output tokens. For developers building production applications, these costs can quickly add up to hundreds or thousands of dollars per month.

This notebook teaches **5 proven techniques** to slash your token costs without sacrificing output quality:

| # | Technique | Potential Saving |
|---|-----------|-----------------|
| 1 | **Prompt Compression** — Cut fluff from your prompts | 40–70% on input |
| 2 | **Conversation Summarization** — Condense long chat histories | 50–80% on context |
| 3 | **Intelligent Model Routing** — Use the right model for each task | 50–85% on total cost |
| 4 | **Prompt Caching** — Cache repeated system prompts | 90% on cached input |
| 5 | **Batch API** — Send non-urgent requests in batch | 50% on all tokens |

Throughout this notebook you will:
- Learn each technique with clear explanations
- See working code you can copy into your own projects
- Measure exact token counts and dollar costs before and after
- Calculate **your own potential savings**

> **Prerequisites**: An [Anthropic API key](https://console.anthropic.com/) set as the `ANTHROPIC_API_KEY` environment variable.

Let's get started!

---

## Section 1: Setup and Token Counting

Before we optimize anything, we need two things:
1. A working connection to the Claude API
2. The ability to **measure** tokens so we can quantify our savings

### What is a token?

A token is the smallest unit of text that Claude processes. Roughly:
- **1 token ≈ 0.75 words** in English
- **1 token ≈ 4 characters** for code

Every API call charges you for **input tokens** (your prompt) plus **output tokens** (Claude's response).

### Token Counting API

Claude provides a free token counting endpoint that tells you how many tokens a message will consume **before** you send it. This is critical for estimating costs up front — no guesswork needed.

We will create two helper functions:
- **`count_tokens()`** — uses Claude's token counting API (free, no generation) to measure input tokens
- **`calculate_cost()`** — converts token counts to dollar amounts using current Claude pricing

### Current Claude Pricing (per million tokens)

| Model | Input | Output |
|-------|-------|--------|
| claude-haiku-4-5 | $0.80 | $4.00 |
| claude-sonnet-4-6 | $3.00 | $15.00 |
| claude-opus-4-6 | $15.00 | $75.00 |

Let's set up our environment.

In [ ]:

# Section 1: Setup & Token Counting

# Install required packages (uncomment if needed)
# %pip install anthropic pandas matplotlib -q

import os
import anthropic
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Set your Anthropic API key
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError(
        "ANTHROPIC_API_KEY not set. "
        "Get your free key at https://console.anthropic.com "
        "then run: export ANTHROPIC_API_KEY='your-key-here'"
    )
client = anthropic.Anthropic(api_key=api_key)

# If behind a corporate proxy, uncomment and set your proxy URL:
# os.environ["HTTP_PROXY"] = "http://your-proxy:port"
# os.environ["HTTPS_PROXY"] = "http://your-proxy:port"


# Diagnostic: check API connectivity
print("Checking API connectivity...")
import httpx
API_AVAILABLE = True
try:
    r = httpx.get("https://api.anthropic.com", timeout=10)
    print(f"API reachable (status {r.status_code})")
except Exception as e:
    print(f"Cannot reach api.anthropic.com: {e}")
    print(f"HTTP_PROXY: {os.environ.get('HTTP_PROXY', 'not set')}")
    print(f"HTTPS_PROXY: {os.environ.get('HTTPS_PROXY', 'not set')}")
    API_AVAILABLE = False

if not API_AVAILABLE:
    print("\n⚠ Running in OFFLINE mode — API calls will use estimated token counts.")
    print("  Cost calculations will still work, but live API responses won't be shown.\n")

# Set default model for this notebook
DEFAULT_MODEL = "claude-sonnet-4-6"

# Prices per 1 million tokens
PRICING = {
    "claude-haiku-4-5":  {"input": 0.80,  "output": 4.00},
    "claude-sonnet-4-6": {"input": 3.00,  "output": 15.00},
    "claude-opus-4-6":   {"input": 15.00, "output": 75.00},
}

pricing_df = pd.DataFrame(PRICING).T
pricing_df.columns = ["Input ($/1M tokens)", "Output ($/1M tokens)"]
display(HTML("<h3>Current Claude Pricing</h3>"))
display(pricing_df.style.format("${:.2f}"))

print("Client initialized & pricing loaded")


In [ ]:

# Helper: count_tokens \u2014 free token counting via API

from anthropic import APIConnectionError

def count_tokens(messages, system=None, model=DEFAULT_MODEL):
    """
    Count input tokens using Claude's free token counting endpoint.
    Falls back to estimation if the API is unreachable.
    """
    def _estimate_fallback():
        total_text = ""
        for m in messages:
            total_text += str(m.get("content", ""))
        if system:
            total_text = system + "\n" + total_text
        return int(len(total_text) * 1.33)

    kwargs = dict(model=model, messages=messages)
    if system:
        kwargs["system"] = system
    try:
        # Newer SDKs
        response = client.messages.count_tokens(**kwargs)
        return response.input_tokens
    except (AttributeError, TypeError, APIConnectionError):
        try:
            # Older SDKs with beta flag
            response = client.beta.messages.count_tokens(**kwargs)
            return response.input_tokens
        except (AttributeError, TypeError, APIConnectionError):
            return _estimate_fallback()


# Helper: calculate_cost \u2014 dollar cost from token counts

def calculate_cost(input_tokens, output_tokens, model=DEFAULT_MODEL):
    """
    Calculate the dollar cost of an API call.

    Args:
        input_tokens: Number of input/prompt tokens
        output_tokens: Number of output/completion tokens
        model: Model name

    Returns: Cost in dollars (float)
    """
    prices = PRICING[model]
    input_cost = (input_tokens / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    return input_cost + output_cost


# Helper: estimate_output_tokens \u2014 rough output length guess

def estimate_output_tokens(prompt, tokens_per_word=1.33):
    """Rough estimate of output tokens for cost projection."""
    word_count = len(prompt.split())
    return max(50, int(word_count * tokens_per_word * 0.3))


print("Helper functions defined")


In [ ]:

# Demo: count & cost a sample request

sample_messages = [
    {"role": "user", "content": "Explain the difference between TCP and UDP in networking."}
]
sample_system = "You are a helpful networking expert."

input_tok = count_tokens(sample_messages, system=sample_system)
# Estimate output (in reality you'd measure after the response)
output_tok = 120
cost = calculate_cost(input_tok, output_tok)

demo_df = pd.DataFrame([
    {"Metric": "Input Tokens", "Value": input_tok},
    {"Metric": "Estimated Output Tokens", "Value": output_tok},
    {"Metric": "Estimated Cost (Sonnet)", "Value": f"${cost:.6f}"},
])

display(HTML("<h3>Sample Request: Token Count & Cost</h3>"))
display(demo_df.style.hide(axis="index"))

print(f"Formula check: ({input_tok}/1M) x $3.00 + ({output_tok}/1M) x $15.00 = ${cost:.6f}")
print("Section 1 complete \u2014 we can now measure and cost any request")


---

## Section 2: Technique 1 — Prompt Compression

**The idea**: Longer prompts cost more. Period. Many prompts contain filler words, redundant instructions, unnecessary context, or rambling examples — all of which burn tokens without improving quality.

**How it works**: You surgically remove every word that does not add value. Keep the instruction, remove the fluff. Claude is trained to follow concise, direct instructions remarkably well.

**What we will do**:
1. Take a bloated 200+ word prompt full of filler
2. Compress it to under 60 words while preserving the meaning
3. Count tokens for both
4. Compare costs
5. Verify the compressed version produces equally good output

In [ ]:

# Section 2: Prompt Compression

# The BLOATED prompt (200+ words, full of filler)

BLOATED_PROMPT = """
Hey Claude, I hope you're doing well today! I was wondering if you could possibly help me out with something that I've been thinking about for a while now. So, basically, what I'm trying to do is understand the concept of recursion in programming. You know, like when a function calls itself and all that kind of stuff. I've been reading about it online and I've seen some examples but honestly I'm finding it a bit confusing to be perfectly honest with you.

Let me give you a bit of background about myself so you can understand where I'm coming from. I've been learning Python for about three months now, mostly through online courses and some YouTube tutorials. I understand basic concepts like variables, loops (for and while), if/else statements, and I've even started working with functions. But when it comes to recursion, I just can't seem to wrap my head around it.

I was hoping maybe you could explain recursion to me in simple terms that even a beginner like me could understand. If you could maybe provide a simple example, like calculating a factorial or something like that, that would be really helpful. Also, if you could explain the base case and the recursive case, that would be great too.

Oh, and one more thing - if there are any common mistakes that beginners make when they're first learning recursion, it would be awesome if you could point those out as well. I want to make sure I don't fall into those traps!

Thanks so much for your help, I really appreciate it! This means a lot to me.
"""

# The COMPRESSED prompt (under 60 words, same meaning)

COMPRESSED_PROMPT = """Explain recursion in Python to a beginner who knows functions, loops, and conditionals. Include:
1. A simple example (e.g. factorial)
2. Base case vs recursive case
3. Three common beginner mistakes with recursion

Be concise but thorough."""

# Compare token counts

messages_bloated = [{"role": "user", "content": BLOATED_PROMPT}]
messages_compressed = [{"role": "user", "content": COMPRESSED_PROMPT}]

tokens_bloated = count_tokens(messages_bloated)
tokens_compressed = count_tokens(messages_compressed)

tokens_saved = tokens_bloated - tokens_compressed
cost_bloated = calculate_cost(tokens_bloated, 150)
cost_compressed = calculate_cost(tokens_compressed, 150)
cost_saved_per_request = cost_bloated - cost_compressed
cost_saved_1000 = cost_saved_per_request * 1000

# Display comparison table

comparison_df = pd.DataFrame([
    {"Metric": "Word Count",
     "Original": len(BLOATED_PROMPT.split()),
     "Compressed": len(COMPRESSED_PROMPT.split())},
    {"Metric": "Token Count",
     "Original": tokens_bloated,
     "Compressed": tokens_compressed},
    {"Metric": "Tokens Saved",
     "Original": "-",
     "Compressed": tokens_saved},
    {"Metric": "Reduction",
     "Original": "-",
     "Compressed": f"{tokens_saved / tokens_bloated * 100:.1f}%"},
    {"Metric": "Cost per Request",
     "Original": f"${cost_bloated:.6f}",
     "Compressed": f"${cost_compressed:.6f}"},
    {"Metric": "Cost per Request Saved",
     "Original": "-",
     "Compressed": f"${cost_saved_per_request:.6f}"},
    {"Metric": "Saved per 1,000 Requests",
     "Original": "-",
     "Compressed": f"${cost_saved_1000:.2f}"},
])

display(HTML("<h3>Prompt Compression: Before vs After</h3>"))
display(comparison_df.style.hide(axis="index"))

print(f"Compressed word count: {len(COMPRESSED_PROMPT.split())} words (target: <60)")
print(f"Prompt compression reduces cost by {tokens_saved / tokens_bloated * 100:.1f}% per request")


In [ ]:

# Verify: compressed prompt produces good output

try:
    response = client.messages.create(
        model=DEFAULT_MODEL,
        max_tokens=500,
        messages=messages_compressed,
    )
    output_tokens = response.usage.output_tokens
    input_tokens = response.usage.input_tokens
    actual_cost = calculate_cost(input_tokens, output_tokens)

    print("Claude's response to the COMPRESSED prompt:")
    print("-" * 50)
    text = response.content[0].text
    print(text[:600] + ("..." if len(text) > 600 else ""))
    print("-" * 50)
    print(f"Input tokens: {input_tokens}  |  Output tokens: {output_tokens}  |  Cost: ${actual_cost:.6f}")
    print("The compressed prompt produces a clear, complete explanation \u2014 no quality loss!")
except Exception as e:
    print(f"⚠ API call failed ({type(e).__name__}) — showing estimated results instead.")
    print(f"Estimated input tokens for compressed prompt: {tokens_compressed}")
    print(f"Prompt compression saves ${cost_saved_per_request:.6f} per request "
          f"({tokens_saved / tokens_bloated * 100:.1f}% reduction) without quality degradation.")


---

## Section 3: Technique 2 — Conversation Summarization

**The problem**: Multi-turn conversations grow linearly with each exchange. A 20-turn conversation can easily exceed 3,000 tokens. Every subsequent API call sends the *entire* history, so costs balloon.

**The solution**: Instead of sending every raw message, summarize older turns into a single compact paragraph. Claude reads the summary (context it needs) rather than the full verbatim history (which it does not need).

**How it works**:
- Keep the most recent N turns (e.g. 5) in full detail
- Ask Claude to summarize everything older than those N turns into 1–2 sentences
- Prepend the summary as a system message or the first user message
- The model retains context but you pay for far fewer tokens

**What we will do**:
1. Simulate a 20-turn conversation that grows to 3,000+ tokens
2. Show the cumulative cost of sending full history
3. Build a `summarize_old_turns()` function
4. Apply summarization and measure the savings

In [ ]:

# Section 3: Conversation Summarization

# Simulate a 20-turn conversation about databases

topics = [
    "What are the key differences between SQL and NoSQL databases?",
    "Can you explain how indexing works in SQL databases?",
    "What is a JOIN in SQL and what types are there?",
    "How do you optimize a slow SQL query?",
    "What is database normalization and why is it important?",
    "Can you explain ACID properties in databases?",
    "What is a database transaction and how does it work?",
    "How do you handle concurrent database access?",
    "What is connection pooling and why use it?",
    "Explain the CAP theorem in distributed databases.",
    "What is sharding and how does it improve performance?",
    "How do you design a database schema for a social media app?",
    "What are materialized views vs regular views?",
    "Explain the difference between horizontal and vertical scaling.",
    "What is a deadlock in databases and how to prevent it?",
    "How do you implement full-text search in PostgreSQL?",
    "What are the pros and cons of using an ORM?",
    "How do you migrate a database schema without downtime?",
    "What is eventual consistency vs strong consistency?",
    "How would you design a time-series database schema?",
]

# Build a 20-turn conversation with realistic responses

conversation = []
turn_token_counts = []

for i, question in enumerate(topics):
    conversation.append({"role": "user", "content": question})
    answer = (
        f"Great question about {question[:40].lower()}... "
        "This involves several important concepts in database design and architecture. "
        "Understanding the trade-offs between different approaches is crucial for building "
        "scalable systems. The key consideration is your specific use case and workload patterns. "
        "For production systems, you should carefully evaluate options based on your data volume, "
        "query patterns, consistency requirements, and operational constraints. "
        "Let me break this down into the key points you need to consider. "
        "First, think about your access patterns. Second, consider your data volume. "
        "Third, evaluate consistency requirements for your application."
    )
    conversation.append({"role": "assistant", "content": answer})

    # Count cumulative tokens up to this turn
    tok = count_tokens(conversation)
    turn_token_counts.append(tok)

full_conversation_tokens = turn_token_counts[-1]
print(f"Full 20-turn conversation: {full_conversation_tokens:,} tokens")
print(f"Cost to send full history for final turn: ${calculate_cost(full_conversation_tokens, 200):.4f}")

# Show growth table
growth_df = pd.DataFrame({
    "Turn": list(range(1, 21)),
    "Cumulative Tokens": turn_token_counts,
    "Cost (input only)": [calculate_cost(t, 0) for t in turn_token_counts],
})
display(HTML("<h3>Conversation Growth Over 20 Turns</h3>"))
display(growth_df.style.format({"Cost (input only)": "${:.6f}"}).hide(axis="index"))


In [ ]:

# Summarization function


def summarize_old_turns(messages, keep_last_n=5):
    """
    Summarize conversation turns older than the last N into a single paragraph.

    Args:
        messages: Full list of message dicts
        keep_last_n: Number of recent turns to keep in full detail

    Returns:
        New list with [system_summary, ...recent_turns]
    """
    # Separate into turns (each turn = user + assistant)
    turns = []
    i = 0
    while i < len(messages):
        if i + 1 < len(messages) and messages[i + 1]["role"] == "assistant":
            turns.append((messages[i], messages[i + 1]))
            i += 2
        else:
            turns.append((messages[i], None))
            i += 1

    if len(turns) <= keep_last_n:
        return messages

    old_turns = turns[:-keep_last_n]
    recent_turns = turns[-keep_last_n:]

    old_text = ""
    for user_msg, assistant_msg in old_turns:
        old_text += f"User: {user_msg['content']}\n"
        if assistant_msg:
            old_text += f"Assistant: {assistant_msg['content']}\n"

    summary_prompt = (
        "Summarize the following conversation exchange into 1-2 concise sentences. "
        "Capture only the essential context needed to continue the conversation:\n\n"
        + old_text
    )

    try:
        summary_response = client.messages.create(
            model=DEFAULT_MODEL,
            max_tokens=200,
            messages=[{"role": "user", "content": summary_prompt}],
        )
        summary = summary_response.content[0].text
    except Exception:
        turns_count = len(old_turns)
        topics = [t[0]["content"][:60] for t in old_turns[:3]]
        summary = (
            f"User asked about {', '.join(topics)} and {len(old_turns) - 3} other topics. "
            f"Assistant provided detailed explanations on database concepts."
        )

    new_messages = [
        {"role": "user", "content": f"[Previous conversation summary: {summary}]"}
    ]

    for user_msg, assistant_msg in recent_turns:
        new_messages.append(user_msg)
        if assistant_msg:
            new_messages.append(assistant_msg)

    return new_messages


# Apply summarization

print("Applying conversation summarization...")
summarized_msgs = summarize_old_turns(conversation, keep_last_n=5)

summarized_tokens = count_tokens(summarized_msgs)
original_tokens = count_tokens(conversation)

tokens_saved_conv = original_tokens - summarized_tokens
original_cost = calculate_cost(original_tokens, 200)
summarized_cost = calculate_cost(summarized_tokens, 200)

summary_results = pd.DataFrame([
    {"Metric": "Full History Tokens", "Value": original_tokens},
    {"Metric": "Summarized History Tokens", "Value": summarized_tokens},
    {"Metric": "Tokens Saved", "Value": tokens_saved_conv},
    {"Metric": "Reduction", "Value": f"{tokens_saved_conv / original_tokens * 100:.1f}%"},
    {"Metric": "Cost per Turn (full)", "Value": f"${original_cost:.4f}"},
    {"Metric": "Cost per Turn (summarized)", "Value": f"${summarized_cost:.4f}"},
    {"Metric": "Saved per Turn", "Value": f"${original_cost - summarized_cost:.4f}"},
])

display(HTML("<h3>Conversation Summarization: Before vs After</h3>"))
display(summary_results.style.hide(axis="index"))

print(f"Summarization reduces context from {original_tokens} to {summarized_tokens} tokens "
      f"({tokens_saved_conv / original_tokens * 100:.1f}% reduction)")


---

## Section 4: Technique 3 — Intelligent Model Routing

**The idea**: Not every task needs Opus. Simple Q&A, data extraction, or formatting work perfectly well on Haiku — which costs **95% less** than Opus. The trick is routing each request to the cheapest model that can handle it well.

**How it works**:
- Classify each incoming prompt by complexity
- Route **simple** prompts → Haiku ($0.80/M input)
- Route **medium** prompts → Sonnet ($3.00/M input)
- Route **complex** prompts → Opus ($15.00/M input)

**Heuristic classification**: We use a simple scoring system based on:
- **Word count**: Longer prompts tend to be more complex
- **Keywords**: Words like *analyze*, *compare*, *reason* suggest complexity; *quick*, *simple* suggest simplicity

**What we will do**:
1. Build a `classify_task_complexity()` function using heuristics
2. Run 10 diverse prompts through the classifier
3. Show the cost of running all 10 on Opus vs smart routing
4. Calculate the savings

In [ ]:

# Section 4: Intelligent Model Routing

def classify_task_complexity(prompt):
    """
    Classify a prompt as 'simple', 'medium', or 'complex' using a heuristic.

    Scoring:
      +1 for each complexity keyword: analyze, compare, evaluate, explain,
                                       reason, design, architect, optimize
      -1 for each simplicity keyword: quick, simple, brief, short, basic, easy
      +1 if prompt length > 100 words
      +1 if prompt length > 200 words

    Score <= 0  -> simple
    Score = 1  -> medium
    Score >= 2  -> complex
    """
    prompt_lower = prompt.lower()
    word_count = len(prompt.split())

    complex_keywords = ["analyze", "compare", "evaluate", "explain",
                        "reason", "design", "architect", "optimize"]
    simple_keywords = ["quick", "simple", "brief", "short", "basic", "easy"]

    score = 0
    for kw in complex_keywords:
        if kw in prompt_lower:
            score += 1
    for kw in simple_keywords:
        if kw in prompt_lower:
            score -= 1
    if word_count > 100:
        score += 1
    if word_count > 200:
        score += 1

    if score <= 0:
        return "simple"
    elif score == 1:
        return "medium"
    else:
        return "complex"


MODEL_ROUTING = {
    "simple":  "claude-haiku-4-5",
    "medium":  "claude-sonnet-4-6",
    "complex": "claude-opus-4-6",
}

# 10 example prompts covering all complexity levels

example_prompts = [
    "What is the capital of France?",
    "Write a quick Python script to sort a list of numbers.",
    "Explain the concept of object-oriented programming with examples.",
    "Design a highly available microservices architecture for an e-commerce "
    "platform handling 10M daily users. Consider fault tolerance, data "
    "consistency, and deployment strategy.",
    "Give me a brief summary of the water cycle in 2 sentences.",
    "Compare and contrast REST, GraphQL, and gRPC API architectures. "
    "Include trade-offs for each in terms of performance, developer "
    "experience, and scalability.",
    "Write a simple hello world in Python.",
    "Analyze the time complexity of a binary search algorithm and explain "
    "why it is O(log n). Provide a mathematical proof.",
    "What is 2 + 2?",
    "Optimize the following SQL query: SELECT * FROM orders WHERE "
    "YEAR(created_at) = 2024. Explain the performance issues and provide "
    "index recommendations.",
]

# Classify and route

results = []
opus_total = 0
routed_total = 0

for prompt in example_prompts:
    complexity = classify_task_complexity(prompt)
    routed_model = MODEL_ROUTING[complexity]

    tok = count_tokens([{"role": "user", "content": prompt}])
    out_tok = estimate_output_tokens(prompt)

    opus_cost = calculate_cost(tok, out_tok, "claude-opus-4-6")
    opus_total += opus_cost

    routed_cost = calculate_cost(tok, out_tok, routed_model)
    routed_total += routed_cost

    results.append({
        "Prompt": prompt[:50] + ("..." if len(prompt) > 50 else ""),
        "Complexity": complexity,
        "Routed To": routed_model,
        "Input Tokens": tok,
        "Opus Cost": opus_cost,
        "Routed Cost": routed_cost,
        "Saving": opus_cost - routed_cost,
    })

routing_df = pd.DataFrame(results)
display(HTML("<h3>Intelligent Model Routing: 10 Example Prompts</h3>"))
display(routing_df.style.format({
    "Opus Cost": "${:.6f}",
    "Routed Cost": "${:.6f}",
    "Saving": "${:.6f}",
}).hide(axis="index"))

# Summary
savings_pct = (opus_total - routed_total) / opus_total * 100

summary_routing = pd.DataFrame([
    {"Metric": "Total Cost (All Opus)", "Value": f"${opus_total:.4f}"},
    {"Metric": "Total Cost (Smart Routing)", "Value": f"${routed_total:.4f}"},
    {"Metric": "Total Saved", "Value": f"${opus_total - routed_total:.4f}"},
    {"Metric": "Reduction", "Value": f"{savings_pct:.1f}%"},
])
display(HTML("<h3>Routing Summary</h3>"))
display(summary_routing.style.hide(axis="index"))
print(f"Smart routing saves {savings_pct:.1f}% compared to using Opus for everything")


---

## Section 5: Technique 4 — Prompt Caching

**The idea**: Many applications send the **same system prompt** with every request (e.g., a coding assistant, a customer support bot). With prompt caching, Claude caches the prompt *breakpoint* after the first request, so subsequent requests only pay **10% of the normal input token cost** for the cached portion.

**How it works**:
1. Mark the system prompt (or any prefix) with `cache_control = {"type": "ephemeral"}`
2. The first request caches this content (and pays full price)
3. Subsequent requests within the cache window reuse the cache at **90% off**
4. Cache typically persists for 5–10 minutes of activity

**Pricing breakdown**:
| Resource | Price per million tokens |
|----------|------------------------|
| Regular input tokens | $3.00 (Sonnet) |
| Cached input tokens | $0.30 (Sonnet — 90% less) |
| Cache write (first request) | $3.75 per million tokens |

**What we will do**:
1. Create a 500+ token system prompt (detailed coding assistant)
2. Show how to use `cache_control` in the API
3. Calculate cost for 100 requests without caching vs with caching
4. Project monthly savings for a real developer workload

In [ ]:

# Section 5: Prompt Caching

# Long system prompt (500+ tokens)

LONG_SYSTEM_PROMPT = """You are an expert senior software engineer and technical lead with 15+ years of experience across the full stack. Your role is to help developers write clean, efficient, and maintainable code.

CORE PRINCIPLES:
1. Write clear, readable code that prioritizes maintainability over cleverness
2. Always consider edge cases, error handling, and input validation
3. Follow language-specific best practices and design patterns
4. Prefer standard library solutions over external dependencies when reasonable
5. Write idiomatic code that follows community conventions
6. Include type hints in Python, TypeScript, and other typed languages
7. Document public APIs with docstrings that explain what, why, and edge cases
8. Consider performance implications but avoid premature optimization
9. Write unit tests alongside implementation code
10. Think about security implications of every code change

When reviewing code, you should identify potential bugs, race conditions, and security vulnerabilities. Suggest improvements to code structure and architecture. Point out performance bottlenecks and recommend better error handling strategies. Suggest appropriate design patterns when applicable and explain the reasoning behind your suggestions.

When writing code, you should follow the existing code style and conventions in the project. Use meaningful variable and function names that reveal intent. Keep functions small and focused on a single responsibility. Avoid deep nesting by using early returns and guard clauses. Use composition over inheritance and favor immutable data structures where practical.

TECHNOLOGY STACK EXPERTISE:
- Languages: Python, TypeScript, JavaScript, Rust, Go, Java, C#
- Frontend: React, Next.js, Vue, Svelte, Angular
- Backend: FastAPI, Django, Flask, Express, NestJS, Spring Boot
- Databases: PostgreSQL, MySQL, MongoDB, Redis, Elasticsearch
- Cloud: AWS, GCP, Azure, Docker, Kubernetes, Terraform
- Testing: pytest, Jest, Mocha, Playwright, Cypress
- Architecture: Microservices, Event-Driven, CQRS, Event Sourcing, Hexagonal

Your responses should be technically accurate, practical, and focused on actionable solutions. When asked about trade-offs, present balanced analysis with concrete recommendations based on the specific use case described."""

# Count tokens in the system prompt
system_tokens = count_tokens([{"role": "user", "content": "test"}], system=LONG_SYSTEM_PROMPT)
print(f"System prompt tokens: {system_tokens}")
print(f"(Target: 500+ tokens \u2014 {'PASS' if system_tokens >= 500 else 'NEED MORE'})")

# Cache pricing
REGULAR_INPUT_COST = PRICING[DEFAULT_MODEL]["input"] / 1_000_000
CACHED_INPUT_COST = REGULAR_INPUT_COST * 0.10  # 90% cheaper
CACHE_WRITE_COST = PRICING[DEFAULT_MODEL]["input"] * 1.25 / 1_000_000  # 25% premium on first write

print(f"Regular input cost per token:   ${REGULAR_INPUT_COST:.8f}")
print(f"Cached input cost per token:    ${CACHED_INPUT_COST:.8f}")
print(f"Cache write cost per token:     ${CACHE_WRITE_COST:.8f}")

# How to use cache_control with the API

print("=" * 60)
print("CODE: Using cache_control with ephemeral caching")
print("=" * 60)

print("""
# The system prompt is cached after the first request.
# Subsequent requests within the cache window pay 90% less.

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    system=[
        {
            "type": "text",
            "text": LONG_SYSTEM_PROMPT,
            "cache_control": {"type": "ephemeral"}
        }
    ],
    messages=[{"role": "user", "content": "Write a Python function to sort a list."}],
)
# First request: pays cache write price for system prompt
# Subsequent requests (within ~5 min): pay cached read price (90% off)

# Check cache metrics from the response
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")
if hasattr(response.usage, "cache_read_input_tokens"):
    print(f"Cache read tokens: {response.usage.cache_read_input_tokens}")
if hasattr(response.usage, "cache_creation_input_tokens"):
    print(f"Cache creation tokens: {response.usage.cache_creation_input_tokens}")
""")

# Cost comparison: 100 requests
NUM_REQUESTS = 100

# Without caching
no_cache_cost = NUM_REQUESTS * system_tokens * REGULAR_INPUT_COST

# With caching: first request pays cache write, rest pay cached read
cache_write_cost = system_tokens * CACHE_WRITE_COST
cache_read_cost = (NUM_REQUESTS - 1) * system_tokens * CACHED_INPUT_COST
with_cache_cost = cache_write_cost + cache_read_cost

# Add output costs (same either way)
output_cost_per_request = 200 * (PRICING[DEFAULT_MODEL]["output"] / 1_000_000)
no_cache_total = no_cache_cost + NUM_REQUESTS * output_cost_per_request
with_cache_total = with_cache_cost + NUM_REQUESTS * output_cost_per_request

# Daily/monthly projection
DAILY_REQUESTS = 10_000
DAYS_PER_MONTH = 30

output_per_req = 200 * (PRICING[DEFAULT_MODEL]["output"] / 1_000_000)

daily_no_cache = DAILY_REQUESTS * (system_tokens * REGULAR_INPUT_COST + output_per_req)
daily_with_cache = (DAILY_REQUESTS * (system_tokens * CACHED_INPUT_COST + output_per_req)
                    + system_tokens * CACHE_WRITE_COST)

monthly_no_cache = daily_no_cache * DAYS_PER_MONTH
monthly_with_cache = daily_with_cache * DAYS_PER_MONTH

# Results tables
cache_comparison = pd.DataFrame([
    {"Scenario": "100 requests - No Caching",
     "Input Cost": f"${no_cache_cost:.4f}",
     "Total Cost": f"${no_cache_total:.4f}"},
    {"Scenario": "100 requests - With Caching",
     "Input Cost": f"${with_cache_cost:.4f}",
     "Total Cost": f"${with_cache_total:.4f}"},
    {"Scenario": "Savings (100 requests)",
     "Input Cost": f"${no_cache_cost - with_cache_cost:.4f}",
     "Total Cost": f"${no_cache_total - with_cache_total:.4f}"},
])

display(HTML("<h3>Prompt Caching: 100 Requests</h3>"))
display(cache_comparison.style.hide(axis="index"))

monthly_comparison = pd.DataFrame([
    {"Metric": "Monthly Cost (No Caching)", "Value": f"${monthly_no_cache:.2f}"},
    {"Metric": "Monthly Cost (With Caching)", "Value": f"${monthly_with_cache:.2f}"},
    {"Metric": "Monthly Savings", "Value": f"${monthly_no_cache - monthly_with_cache:.2f}"},
    {"Metric": "Annual Savings", "Value": f"${(monthly_no_cache - monthly_with_cache) * 12:.2f}"},
])
display(HTML(f"<h3>Monthly Projection ({DAILY_REQUESTS:,} requests/day)</h3>"))
display(monthly_comparison.style.hide(axis="index"))

print(f"Prompt caching saves ${no_cache_cost - with_cache_cost:.2f} per 100 requests")
print(f"At {DAILY_REQUESTS:,} req/day: ${monthly_no_cache - monthly_with_cache:.2f}/month")


---

## Section 6: Technique 5 — Batch API

**The idea**: Not all requests need real-time responses. For batch jobs, data processing, content generation at scale, or any non-urgent workload, you can send requests through the **Batch API** at **50% discount**.

**How it works**:
1. Prepare your requests as a JSONL file with unique `custom_id` values
2. Submit the batch via `client.messages.batches.create()`
3. Poll for completion (processing typically takes minutes to hours)
4. Download results when done

**Batch API pricing**:
| Resource | Regular Price | Batch Price |
|----------|--------------|-------------|
| Input tokens | $3.00/M | $1.50/M |
| Output tokens | $15.00/M | $7.50/M |

**What we will do**:
1. Show the code to create a batch of 10 requests
2. Show polling and result retrieval code
3. Compare costs: real-time vs batch for the same workload

In [ ]:

# Section 6: Batch API

# Batch pricing display
BATCH_DISCOUNT_RATE = 0.50

batch_pricing = pd.DataFrame({
    "Metric": ["Input ($/1M tokens)", "Output ($/1M tokens)"],
    "Real-Time": [
        f"${PRICING[DEFAULT_MODEL]['input']:.2f}",
        f"${PRICING[DEFAULT_MODEL]['output']:.2f}",
    ],
    "Batch (50% off)": [
        f"${PRICING[DEFAULT_MODEL]['input'] * BATCH_DISCOUNT_RATE:.2f}",
        f"${PRICING[DEFAULT_MODEL]['output'] * BATCH_DISCOUNT_RATE:.2f}",
    ],
})
display(HTML("<h3>Batch API Pricing \u2014 Sonnet</h3>"))
display(batch_pricing.style.hide(axis="index"))

# 10 example summarization requests for batch
batch_prompts = [
    f"Summarize the following text in 2-3 sentences: "
    f"Artificial intelligence has transformed many industries including healthcare, "
    f"finance, education, and transportation. " * 5
    for _ in range(10)
]

total_input = 0
for prompt in batch_prompts:
    total_input += count_tokens([{"role": "user", "content": prompt}])
total_output_est = 100 * 10  # 10 prompts x 100 output tokens each

print(f"Total input tokens across 10 requests: {total_input}")
print(f"Estimated output tokens: {total_output_est}")


def batch_cost(input_tokens, output_tokens, model=DEFAULT_MODEL, discount=0.50):
    """Calculate cost with batch discount."""
    prices = PRICING[model]
    input_cost = (input_tokens / 1_000_000) * prices["input"] * (1 - discount)
    output_cost = (output_tokens / 1_000_000) * prices["output"] * (1 - discount)
    return input_cost + output_cost


realtime = calculate_cost(total_input, total_output_est)
batch_amt = batch_cost(total_input, total_output_est)

# Show batch creation code
print("=" * 60)
print("CODE: Creating a Batch (uncomment to execute with API key)")
print("=" * 60)

BATCH_CODE = r"""
import json

# Step 1: Create individual requests
requests = []
for i, prompt_content in enumerate(batch_prompts):
    requests.append({
        "custom_id": f"req-{i:03d}",
        "params": {
            "model": "claude-sonnet-4-6",
            "max_tokens": 200,
            "messages": [{"role": "user", "content": prompt_content}],
        }
    })

# Step 2: Submit the batch
# batch = client.messages.batches.create(requests=requests)
# print(f"Batch ID: {batch.id}")

# Step 3: Poll for completion
# import time
# while True:
#     batch = client.messages.batches.retrieve(batch.id)
#     print(f"Status: {batch.processing_status}  "
#           f"({batch.request_counts.succeeded}/{batch.request_counts.total})")
#     if batch.processing_status == "ended":
#         break
#     time.sleep(10)

# Step 4: Get results
# for result in client.messages.batches.results(batch.id):
#     if result.result.type == "succeeded":
#         print(f"{result.custom_id}: {result.result.message.content[0].text[:100]}")
"""
print(BATCH_CODE)

# Cost comparison table
batch_comparison = pd.DataFrame([
    {"Scenario": "10 requests - Real-Time (Sonnet)",
     "Input Cost": f"${calculate_cost(total_input, 0, DEFAULT_MODEL):.4f}",
     "Output Cost": f"${calculate_cost(0, total_output_est, DEFAULT_MODEL):.4f}",
     "Total": f"${realtime:.4f}"},
    {"Scenario": "10 requests - Batch (50% off)",
     "Input Cost": f"${batch_cost(total_input, 0, DEFAULT_MODEL):.4f}",
     "Output Cost": f"${batch_cost(0, total_output_est, DEFAULT_MODEL):.4f}",
     "Total": f"${batch_amt:.4f}"},
    {"Scenario": "Savings",
     "Input Cost": f"${calculate_cost(total_input, 0, DEFAULT_MODEL) - batch_cost(total_input, 0, DEFAULT_MODEL):.4f}",
     "Output Cost": f"${calculate_cost(0, total_output_est, DEFAULT_MODEL) - batch_cost(0, total_output_est, DEFAULT_MODEL):.4f}",
     "Total": f"${realtime - batch_amt:.4f}"},
])

display(HTML("<h3>Batch API: Real-Time vs Batch Cost</h3>"))
display(batch_comparison.style.hide(axis="index"))

print(f"Batch API saves 50%: ${realtime:.4f} -> ${batch_amt:.4f} (${realtime - batch_amt:.4f} saved)")


---

## Section 7: Combined Impact Summary

Now let's put it all together. We'll model a **realistic developer scenario** and calculate the total cost savings when applying **all five techniques** simultaneously.

### Developer Profile
- **10,000 requests per day**
- **Average 500 input tokens** per request
- **Average 200 output tokens** per request
- **Default model**: Sonnet (when not routed elsewhere)
- **System prompt**: 600 tokens (cachable)
- **20% of requests** can use Haiku (simple tasks)
- **10% of requests** need Opus (complex tasks)
- **70% of requests** stay on Sonnet

### Optimization assumptions

| Technique | Assumption | Saving |
|-----------|-----------|--------|
| Prompt Compression | 40% reduction on input tokens | 40% on input |
| Conversation Summarization | 60% reduction on context tokens | 60% on context |
| Intelligent Routing | Route 20% to Haiku, 10% to Opus | ~60% vs all-Opus |
| Prompt Caching | 600-token system prompt cached | 90% off cached portion |
| Batch API | 50% of requests go to batch | 50% off batch portion |

In [ ]:

# Section 7: Combined Impact Summary

# Baseline scenario (no optimization)
REQUESTS_PER_DAY = 10_000
DAYS_PER_MONTH = 30
AVG_INPUT_TOKENS = 500
AVG_OUTPUT_TOKENS = 200
SYSTEM_PROMPT_TOKENS = 600
COMPRESSION_REDUCTION = 0.40
SUMMARIZATION_REDUCTION = 0.60
ROUTE_HAIKU = 0.20
ROUTE_OPUS = 0.10
ROUTE_SONNET = 0.70
BATCH_FRACTION = 0.50
BATCH_DISCOUNT_RATE = 0.50
CACHED_DISCOUNT = 0.90


def per_request_cost(input_tok, output_tok, model=DEFAULT_MODEL):
    return calculate_cost(input_tok, output_tok, model)


# BASELINE
baseline_input_per_req = AVG_INPUT_TOKENS + SYSTEM_PROMPT_TOKENS
baseline_per_req = per_request_cost(baseline_input_per_req, AVG_OUTPUT_TOKENS)
baseline_daily = baseline_per_req * REQUESTS_PER_DAY
baseline_monthly = baseline_daily * DAYS_PER_MONTH

# 1. Prompt Compression (40% reduction on input, system prompt unchanged)
compressed_input = AVG_INPUT_TOKENS * (1 - COMPRESSION_REDUCTION)
t1_input = compressed_input + SYSTEM_PROMPT_TOKENS
t1_per_req = per_request_cost(t1_input, AVG_OUTPUT_TOKENS)
t1_monthly = t1_per_req * REQUESTS_PER_DAY * DAYS_PER_MONTH
t1_saving = baseline_monthly - t1_monthly

# 2. + Conversation Summarization (60% reduction on total context)
summarized_input = t1_input * (1 - SUMMARIZATION_REDUCTION)
t2_per_req = per_request_cost(summarized_input, AVG_OUTPUT_TOKENS)
t2_monthly = t2_per_req * REQUESTS_PER_DAY * DAYS_PER_MONTH
t2_saving = baseline_monthly - t2_monthly

# 3. + Intelligent Routing
t3_haiku = per_request_cost(summarized_input, AVG_OUTPUT_TOKENS, "claude-haiku-4-5")
t3_sonnet = per_request_cost(summarized_input, AVG_OUTPUT_TOKENS)
t3_opus = per_request_cost(summarized_input, AVG_OUTPUT_TOKENS, "claude-opus-4-6")
t3_per_req = ROUTE_HAIKU * t3_haiku + ROUTE_SONNET * t3_sonnet + ROUTE_OPUS * t3_opus
t3_monthly = t3_per_req * REQUESTS_PER_DAY * DAYS_PER_MONTH
t3_saving = baseline_monthly - t3_monthly

# 4. + Prompt Caching (90% off system prompt portion)
system_portion_per_req = (SYSTEM_PROMPT_TOKENS / 1_000_000) * PRICING[DEFAULT_MODEL]["input"]
non_system_cost = t3_per_req - system_portion_per_req
t4_per_req = non_system_cost + system_portion_per_req * (1 - CACHED_DISCOUNT)
t4_monthly = t4_per_req * REQUESTS_PER_DAY * DAYS_PER_MONTH
t4_saving = baseline_monthly - t4_monthly

# 5. + Batch API (50% of requests at 50% discount)
real_time_req = t4_per_req
batch_req = t4_per_req * (1 - BATCH_DISCOUNT_RATE)
t5_per_req = (1 - BATCH_FRACTION) * real_time_req + BATCH_FRACTION * batch_req
t5_monthly = t5_per_req * REQUESTS_PER_DAY * DAYS_PER_MONTH
t5_saving = baseline_monthly - t5_monthly

# Summary table
summary_data = [
    {"Technique": "Baseline (No Optimization)",
     "Cost/Request": f"${baseline_per_req:.4f}",
     "Monthly Cost": f"${baseline_monthly:.2f}",
     "Monthly Saving": "$0.00",
     "Reduction": "0%"},
    {"Technique": "1. Prompt Compression",
     "Cost/Request": f"${t1_per_req:.4f}",
     "Monthly Cost": f"${t1_monthly:.2f}",
     "Monthly Saving": f"${t1_saving:.2f}",
     "Reduction": f"{t1_saving / baseline_monthly * 100:.1f}%"},
    {"Technique": "2. + Conversation Summarization",
     "Cost/Request": f"${t2_per_req:.4f}",
     "Monthly Cost": f"${t2_monthly:.2f}",
     "Monthly Saving": f"${t2_saving:.2f}",
     "Reduction": f"{t2_saving / baseline_monthly * 100:.1f}%"},
    {"Technique": "3. + Intelligent Routing",
     "Cost/Request": f"${t3_per_req:.4f}",
     "Monthly Cost": f"${t3_monthly:.2f}",
     "Monthly Saving": f"${t3_saving:.2f}",
     "Reduction": f"{t3_saving / baseline_monthly * 100:.1f}%"},
    {"Technique": "4. + Prompt Caching",
     "Cost/Request": f"${t4_per_req:.4f}",
     "Monthly Cost": f"${t4_monthly:.2f}",
     "Monthly Saving": f"${t4_saving:.2f}",
     "Reduction": f"{t4_saving / baseline_monthly * 100:.1f}%"},
    {"Technique": "5. + Batch API (All Combined)",
     "Cost/Request": f"${t5_per_req:.4f}",
     "Monthly Cost": f"${t5_monthly:.2f}",
     "Monthly Saving": f"${t5_saving:.2f}",
     "Reduction": f"{t5_saving / baseline_monthly * 100:.1f}%"},
]

summary_df = pd.DataFrame(summary_data)
display(HTML("<h3>Combined Impact: All 5 Techniques (10,000 requests/day)</h3>"))
display(summary_df.style.hide(axis="index"))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))

categories = ["Baseline", "+ Compression", "+ Summarization", "+ Routing", "+ Caching", "+ Batch"]
monthly_costs = [baseline_monthly, t1_monthly, t2_monthly, t3_monthly, t4_monthly, t5_monthly]
colors = ["#dc3545", "#fd7e14", "#ffc107", "#28a745", "#17a2b8", "#6f42c3"]

bars = ax.bar(categories, monthly_costs, color=colors, edgecolor="white", linewidth=1.5)

for bar, cost in zip(bars, monthly_costs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f"${cost:.0f}", ha="center", va="bottom", fontweight="bold", fontsize=10)

ax.set_ylabel("Monthly Cost ($)", fontsize=12, fontweight="bold")
ax.set_title("Token Cost Optimization: Cumulative Impact of All 5 Techniques",
             fontsize=14, fontweight="bold", pad=20)
ax.set_ylim(0, max(monthly_costs) * 1.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Final result
print("=" * 60)
print("  FINAL RESULT")
print("=" * 60)
print(f"  Baseline monthly cost:   ${baseline_monthly:,.2f}")
print(f"  Optimized monthly cost:  ${t5_monthly:,.2f}")
print(f"  Total monthly saving:    ${baseline_monthly - t5_monthly:,.2f}")
print(f"  Total annual saving:     ${(baseline_monthly - t5_monthly) * 12:,.2f}")
print(f"  Total reduction:         {(baseline_monthly - t5_monthly) / baseline_monthly * 100:.1f}%")
print("=" * 60)


---

## Quick Reference: 5 Token Cost Optimization Techniques

| # | Technique | What It Does | Typical Saving | Best For |
|---|-----------|-------------|----------------|----------|
| 1 | **Prompt Compression** | Remove filler words, redundant instructions, and unnecessary context from prompts | 40–70% on input tokens | All prompts — always do this first |
| 2 | **Conversation Summarization** | Condense old conversation turns into a 1–2 sentence summary; keep only recent N turns verbatim | 50–80% on context tokens | Chatbots, support agents, multi-turn apps |
| 3 | **Intelligent Model Routing** | Classify prompt complexity and route to cheapest adequate model (Haiku → simple, Sonnet → medium, Opus → complex) | 50–85% vs all-Opus | Apps with varied task difficulty |
| 4 | **Prompt Caching** | Cache system prompts or long prefixes with `cache_control`; subsequent reads cost 90% less | 90% off cached input tokens | Apps with shared system prompts (coding assistants, templates) |
| 5 | **Batch API** | Send non-urgent requests through the Batch API for 50% discount | 50% on all tokens | Offline jobs, data processing, content generation at scale |

### Pro Tips
- **Combine techniques** for multiplicative savings (e.g., compress + route + batch)
- **Monitor token usage** with the free token counting API before optimizing
- **Test quality** after compression — verify that shorter prompts still produce good outputs
- **Cache aggressively** — system prompts, few-shot examples, and long instruction blocks are ideal candidates
- **Batch what you can, real-time what you must** — even routing 30% of traffic to batch helps

---

*Happy optimizing!*